# Lesson 18 Lab — Serving INT4 with vLLM

**Puzzle:** If a checkpoint says AWQ or GPTQ, will vLLM necessarily run it efficiently on the current GPU?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Serving performance belongs to a runtime, not to a checkpoint label. vLLM combines quantized linear kernels with scheduling, continuous batching, paged KV cache, prefix caching, and a request distribution. A PyTorch microbenchmark can warn about shape sensitivity, but it cannot stand in for requests-per-second or latency percentiles from a vLLM server.


## 0. Predict before running

1. Separate checkpoint-format support, hardware support, kernel dispatch, and service-load performance.
2. Predict whether the reference W4 dequantized matrix path wins at every tested batch.
3. Design a serving workload that reports TTFT and inter-token latency separately.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A vLLM service couples checkpoint format, quantization backend, model runner, scheduler, paged KV cache, CUDA graphs, request batching, and sampling. Linear-kernel latency is only one component.

- vLLM selects quantization kernels through a changing model-format and hardware compatibility matrix.
- Serving performance includes scheduling, KV cache, batching, and request distribution—not only linear layers.
- An import probe cannot replace a server benchmark.


## 2. Derive the mechanism

Prefill cost grows with prompt work while decode repeatedly processes small token steps and reads KV cache. Continuous batching improves utilization by combining requests, but queueing changes time-to-first-token and tail latency.

Prefill and Decode produce different matrix shapes and interact differently with batching. Service throughput also depends on arrival rate, prompt/output lengths, scheduler policy, cache capacity, and queueing. A weight-only checkpoint that loads successfully can still fall back to a slow kernel for some layers or lose its memory benefit to KV cache at long context.

The acceptance chain is format metadata → model load → quantized module/operator trace → output quality → controlled request workload → latency/throughput/capacity. An import probe only reaches the first compatibility edge.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "18-vllm-int4-serving"
device = require_cuda()
torch.manual_seed(2026 + 18)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | BF16 PyTorch matrix path for batches 1, 8, and 32 |
| Candidate | reference dequantized W4 matrix path at the same shapes |
| Held constant | weight/input shapes, GPU, warm-up, repetitions; no server or scheduler |
| Measurements | operator median/p90 by batch plus vLLM installation and service-benchmark status |
| Evidence | `compatibility-probe` |

**Experiment:** Probe vLLM availability and benchmark a small PyTorch W4-dequantized matmul across batch sizes as a backend-independent shape warning.


## 5. Read the experiment code

The lab records vLLM availability and uses PyTorch batch-shape timings only as a warning; it labels vLLM service throughput `not_measured`.

The notebook probes vLLM availability, then runs a backend-independent PyTorch shape experiment. The W4 candidate is a dequantized reference tensor, so it tests how the resulting matrix shape behaves—not vLLM's AWQ/GPTQ kernel. Results are stored under `pytorch_shape_warning` to make that boundary visible.

A true service cell would start a server, wait for readiness, issue a frozen request trace, collect TTFT/ITL/latency percentiles and throughput, then terminate cleanly. None of that is synthesized here.

Only after these variables match the protocol should the cell be executed.


In [2]:
import importlib.util
w=torch.randn(2048,2048,device=device,dtype=torch.bfloat16); _,_,dq=symmetric_quantize(w,bits=4,group_size=128); dq=dq.bfloat16()
rows=[]
for batch in (1,8,32):
    x=torch.randn(batch,2048,device=device,dtype=torch.bfloat16)
    rows.append({"batch":batch,"bf16":cuda_benchmark(lambda:x@w.t(),warmup=4,repeats=15),
                 "reference_w4_dequant":cuda_benchmark(lambda:x@dq.t(),warmup=4,repeats=15)})
installed=importlib.util.find_spec("vllm") is not None
result=base_result(18,"compatibility-probe"); result.update({"vllm_installed":installed,"pytorch_shape_warning":rows,
    "vllm_service_benchmark":"not_measured","conclusion":"PyTorch shape timing was measured separately; vLLM service performance requires an installed server and load test."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| vLLM installed | no |
| Service benchmark | not_measured |
| Batch 1 BF16 median | 0.019520 ms |
| Batch 1 reference W4 median | 0.019424 ms |
| Batch 32 BF16 median | 0.018976 ms |
| Batch 32 reference W4 median | 0.019072 ms |


## 7. Interpret rather than merely print

The tiny matrix probe produced nearly tied medians: at batch 1, BF16 was 0.019520 ms and the reference W4-dequant tensor 0.019424 ms; at batch 8 they were 0.019168 and 0.018912 ms; at batch 32 the candidate reversed slightly to 0.019072 versus 0.018976 ms. vLLM was not installed and service performance is explicitly `not_measured`.

Sub-microsecond differences of this kind are not a serving result. They show that shape can reverse a small operator comparison and reinforce why a full request workload is needed.

**Inspection rule:** The timing is labeled PyTorch GPU evidence. vLLM throughput remains `not_measured` when the package/server is absent.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The named optional backend did not complete a native run in this environment. Package and failure evidence are retained; service or kernel performance is not inferred.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "PyTorch shape timing was measured separately; vLLM service performance requires an installed server and load test.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "compatibility-probe",
  "executed_at_utc": "2026-08-07T14:45:55+00:00",
  "lesson": 18,
  "pytorch_shape_warning": [
    {
      "batch": 1,
      "bf16": {
        "median_ms": 0.01952,
        "p90_ms": 0.02032,
        "repeats": 15,
        "samples_ms": [
          0.030528,
          0.028448,
          0.02032,
          0.019488,
          0.01984,
          0.01968,
          0.019488,
          0.019616,
          0.019456,
          0.01952,
          0.019872,
          0.0192,
          0.01904,
          0.019168,
          0.019488
        ],
        "warmup": 4
      },
      "reference_w4_dequant": {
        "median_m

## 9. Make the bounded decision

> Pass checkpoint-format, hardware, load, operator, quality, and service-load gates before adopting a vLLM INT4 path.

**Acceptance/rollback:** Pass format/hardware load, operator, quality, TTFT, TPOT/inter-token latency, throughput, p90/p99, peak memory, and sustained-concurrency gates with a frozen request distribution.

**Failure analysis:** Reporting this table as vLLM speed would mislabel the backend and ignore scheduling. Other traps are benchmarking one warm cache prompt, mixing different model revisions, omitting output length, and comparing throughput at unequal latency or quality. Quantization compatibility matrices also change across versions, so the exact release must be pinned.


## 10. Extend the evidence

Install a supported vLLM release in a separate environment, load one documented AWQ or GPTQ model, confirm module/operator selection, and run `vllm bench serve` with fixed prompt/output distributions and concurrency. Report TTFT p50/p95, ITL, end-to-end latency, tokens/s, GPU memory, and rejected requests.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
